In [ ]:
"""
==================================================
ML LEARNING JOURNEY - DAY 108
==================================================
Week: 16 of 24
Day: 108 of 168
Date: May 27, 2026
Topic: SLAM & Mapping

Week 16 Progress:
✅ Day 106: Gazebo Setup & World Building
✅ Day 107: TurtleBot3 Sensors & Teleoperation
🔄 Day 108: SLAM & Mapping (TODAY!)
⬜ Day 109: TBD
⬜ Day 110: TBD
⬜ Day 111: TBD
⬜ Day 112: TBD

Progress: 28.6% (2/7 days)

==================================================
🎯 Week 16 Project: Gazebo + TurtleBot
- Install Gazebo Garden and ROS2-Gazebo bridge
- Launch and build custom Gazebo worlds
- Set up TurtleBot3 simulation
- Control TurtleBot with keyboard and sensors
- Visualize camera and LiDAR data in RViz

🎯 Today's Learning Objectives:
1. Understand SLAM concepts and why it matters
2. Run slam_toolbox with TurtleBot3 in Gazebo
3. Build a map by driving the robot around
4. Save and load the generated map
5. Understand occupancy grid maps

📚 Today's Structure:
   Part 1 (2h): SLAM Fundamentals
   Part 2 (2h): Building a Map with slam_toolbox
   Part 3 (1h): Saving, Loading & Analyzing Maps

🎯 SUCCESS CRITERIA:
   ✅ Understand what SLAM is and why odometry alone isn't enough
   ✅ Launch slam_toolbox with TurtleBot3
   ✅ Drive around and build a complete map
   ✅ Save the map as .pgm and .yaml files
   ✅ Reload the saved map successfully

==================================================
"""

In [1]:
print("=" * 80)
print("🗺️  SLAM & MAPPING — Day 108")
print("=" * 80)
print("""
Today we tackle one of robotics' biggest challenges:
How does a robot figure out WHERE it is while simultaneously
building a map of WHERE IT HAS BEEN?

This is SLAM — Simultaneous Localization and Mapping.
It's the foundation of all autonomous navigation.
""")
print("=" * 80)

🗺️  SLAM & MAPPING — Day 108

Today we tackle one of robotics' biggest challenges:
How does a robot figure out WHERE it is while simultaneously
building a map of WHERE IT HAS BEEN?

This is SLAM — Simultaneous Localization and Mapping.
It's the foundation of all autonomous navigation.



In [2]:
print("\n" + "=" * 80)
print("🧠 PART 1: SLAM FUNDAMENTALS")
print("=" * 80)


🧠 PART 1: SLAM FUNDAMENTALS


In [3]:
# ==================================================
# EXERCISE 1.1: WHAT IS SLAM?
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.1: What is SLAM?")
print("=" * 80)

"""
📖 THEORY: SLAM

SLAM = Simultaneous Localization and Mapping

The chicken-and-egg problem of robotics:
- To build a map, you need to know WHERE you are
- To know where you are, you need a MAP

SLAM solves both problems at the same time using:
- Sensor data (LiDAR, camera)
- Motion estimates (odometry, IMU)
- Probabilistic math (particle filters, graph optimization)
"""

print("""
📋 WHY ODOMETRY ALONE ISN'T ENOUGH:
─────────────────────────────────────────────────────────────

  Odometry = estimating position from wheel rotations.
  Problem: errors accumulate over time (wheel slip, surface variation).

  After driving 10 meters:
  ├── Ideal: robot knows it's exactly 10m away
  └── Reality: robot thinks it's 9.7m away at a slightly wrong angle

  After driving 100 meters:
  ├── Odometry error can be 1-2 meters off
  └── Map would be completely distorted

  SLAM fixes this by:
  1. Building a map from LiDAR readings
  2. Comparing current LiDAR to the map (loop closure)
  3. Correcting accumulated errors when it recognizes a place


📋 SLAM PIPELINE:
─────────────────────────────────────────────────────────────

  LiDAR scan ──▶ Scan Matching ──▶ Position estimate
       ↑               │                    │
       │               ▼                    ▼
  Odometry ──▶ Motion Model ──▶  Map Update
                                      │
                              Loop Closure Detection
                              (seen this place before?)
                                      │
                              Pose Graph Optimization
                              (correct all past errors)


📋 TYPES OF SLAM:
─────────────────────────────────────────────────────────────

  2D SLAM (what we use today):
  ├── Uses 2D LiDAR scan
  ├── Builds 2D occupancy grid map
  ├── Works for flat environments (warehouses, homes)
  └── slam_toolbox, gmapping, cartographer

  3D SLAM:
  ├── Uses 3D LiDAR or depth camera
  ├── Builds 3D point cloud map
  ├── Works for complex 3D environments
  └── RTAB-Map, LIO-SAM, LeGO-LOAM

  Visual SLAM:
  ├── Uses camera only (no LiDAR)
  ├── Feature detection and matching
  └── ORB-SLAM3, VINS-Mono


📋 OCCUPANCY GRID MAP:
─────────────────────────────────────────────────────────────

  The output of 2D SLAM is an occupancy grid:
  ├── Grid of cells (typically 5cm x 5cm each)
  ├── Each cell has a value:
  │   ├── 0   = free space (white)
  │   ├── 100 = occupied (black wall/obstacle)
  │   └── -1  = unknown (grey, not yet explored)
  └── Saved as a .pgm image + .yaml metadata file

  This map is what Nav2 (navigation stack) uses to plan paths!
""")

print("\n✅ Exercise 1.1 Complete!")
print("=" * 80)


EXERCISE 1.1: What is SLAM?

📋 WHY ODOMETRY ALONE ISN'T ENOUGH:
─────────────────────────────────────────────────────────────

  Odometry = estimating position from wheel rotations.
  Problem: errors accumulate over time (wheel slip, surface variation).

  After driving 10 meters:
  ├── Ideal: robot knows it's exactly 10m away
  └── Reality: robot thinks it's 9.7m away at a slightly wrong angle

  After driving 100 meters:
  ├── Odometry error can be 1-2 meters off
  └── Map would be completely distorted

  SLAM fixes this by:
  1. Building a map from LiDAR readings
  2. Comparing current LiDAR to the map (loop closure)
  3. Correcting accumulated errors when it recognizes a place


📋 SLAM PIPELINE:
─────────────────────────────────────────────────────────────

  LiDAR scan ──▶ Scan Matching ──▶ Position estimate
       ↑               │                    │
       │               ▼                    ▼
  Odometry ──▶ Motion Model ──▶  Map Update
                                      

In [4]:
# ==================================================
# EXERCISE 1.2: SLAM TOOLBOX
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.2: slam_toolbox Overview")
print("=" * 80)

"""
📖 THEORY: slam_toolbox

slam_toolbox is the recommended SLAM package for ROS2.
It's what TurtleBot3's navigation stack uses by default.

Features:
- Online SLAM (build map while driving)
- Lifelong mapping (update existing map)
- Map serialization (save/load)
- Loop closure detection
- Works great with 2D LiDAR
"""

print("""
📋 SLAM TOOLBOX MODES:
─────────────────────────────────────────────────────────────

  online_async_launch.py   → Build map in real-time (what we use)
  online_sync_launch.py    → Synchronous (slower but more accurate)
  localization_launch.py   → Localize in existing map (no new mapping)
  lifelong_launch.py       → Update existing map with new data


📋 SLAM TOOLBOX TOPICS:
─────────────────────────────────────────────────────────────

  Subscribes to:
  ├── /scan     → LiDAR data (input)
  └── /tf       → Transform tree (odom → base_link)

  Publishes:
  ├── /map      → Occupancy grid map (nav_msgs/OccupancyGrid)
  ├── /map_metadata → Map info (resolution, size, origin)
  └── /tf       → map → odom transform (output!)

  Services:
  ├── /slam_toolbox/save_map     → save map to disk
  ├── /slam_toolbox/serialize_map → save full SLAM state
  └── /slam_toolbox/deserialize_map → load SLAM state


📋 INSTALL SLAM TOOLBOX:
─────────────────────────────────────────────────────────────
  # Should already be installed from Day 107, but verify:
  sudo apt install -y ros-humble-slam-toolbox

  # Verify:
  ros2 pkg list | grep slam
  # Expected: slam_toolbox
""")

print("\n✅ Exercise 1.2 Complete!")
print("=" * 80)


EXERCISE 1.2: slam_toolbox Overview

📋 SLAM TOOLBOX MODES:
─────────────────────────────────────────────────────────────

  online_async_launch.py   → Build map in real-time (what we use)
  online_sync_launch.py    → Synchronous (slower but more accurate)
  localization_launch.py   → Localize in existing map (no new mapping)
  lifelong_launch.py       → Update existing map with new data


📋 SLAM TOOLBOX TOPICS:
─────────────────────────────────────────────────────────────

  Subscribes to:
  ├── /scan     → LiDAR data (input)
  └── /tf       → Transform tree (odom → base_link)

  Publishes:
  ├── /map      → Occupancy grid map (nav_msgs/OccupancyGrid)
  ├── /map_metadata → Map info (resolution, size, origin)
  └── /tf       → map → odom transform (output!)

  Services:
  ├── /slam_toolbox/save_map     → save map to disk
  ├── /slam_toolbox/serialize_map → save full SLAM state
  └── /slam_toolbox/deserialize_map → load SLAM state


📋 INSTALL SLAM TOOLBOX:
──────────────────────────────

In [5]:
print("\n" + "=" * 80)
print("🗺️  PART 2: BUILDING A MAP WITH SLAM_TOOLBOX")
print("=" * 80)


🗺️  PART 2: BUILDING A MAP WITH SLAM_TOOLBOX


In [6]:
# ==================================================
# EXERCISE 2.1: LAUNCHING SLAM
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.1: Launching SLAM with TurtleBot3")
print("=" * 80)

print("""
📋 COMPLETE SLAM SETUP (4 terminals):
─────────────────────────────────────────────────────────────

  Terminal 1: Launch TurtleBot3 in world with obstacles
  ──────────────────────────────────────────────────────
  export TURTLEBOT3_MODEL=burger
  ros2 launch turtlebot3_gazebo turtlebot3_world.launch.py

  ⏱️  Wait until Gazebo fully loads (robot visible in world)


  Terminal 2: Launch slam_toolbox
  ──────────────────────────────────────────────────────
  ros2 launch slam_toolbox online_async_launch.py \\
    slam_params_file:=/opt/ros/humble/share/turtlebot3_cartographer/config/turtlebot3_lds_2d.lua \\
    use_sim_time:=true

  # Simpler version (uses default params):
  ros2 launch slam_toolbox online_async_launch.py \\
    use_sim_time:=true

  # use_sim_time:=true = use Gazebo clock, not wall clock
  # CRITICAL: always set this for simulation!


  Terminal 3: Launch RViz2 with SLAM config
  ──────────────────────────────────────────────────────
  ros2 launch nav2_bringup rviz_launch.py

  # Or manually open RViz2 and add:
  rviz2

  In RViz2, add these displays:
  ├── Map (Topic: /map)           ← the SLAM map being built
  ├── LaserScan (Topic: /scan)    ← current LiDAR reading
  ├── RobotModel                  ← TurtleBot3 model
  └── TF                          ← coordinate frames

  Set Fixed Frame to: map


  Terminal 4: Keyboard teleoperation
  ──────────────────────────────────────────────────────
  ros2 run turtlebot3_teleop teleop_keyboard


📋 BUILDING THE MAP:
─────────────────────────────────────────────────────────────

  Strategy for good map coverage:
  1. Drive slowly (linear: 0.1-0.15 m/s)
  2. Drive along walls to map boundaries
  3. Cover all open areas systematically
  4. Return to starting point (loop closure!)
  5. Drive through already-mapped areas to refine

  Watch in RViz2:
  ├── Grey areas = unexplored
  ├── White areas = free space (robot can drive here)
  ├── Black areas = walls/obstacles
  └── Map grows as you explore! ✅

  Signs of good SLAM:
  ✅ Map updates smoothly as you drive
  ✅ Walls appear as clean black lines
  ✅ Free space is clearly white
  ✅ Loop closure: map snaps into alignment when revisiting

  Signs of poor SLAM:
  ❌ Walls appear doubled/blurry → driving too fast
  ❌ Map doesn't update → use_sim_time not set correctly
  ❌ Robot position jumps → loop closure correction (normal!)


📋 SLAM PARAMS (optional tuning):
─────────────────────────────────────────────────────────────
  # Default params work well for TurtleBot3
  # If map quality is poor, create mapper_params.yaml:

  slam_toolbox:
    ros__parameters:
      # Solver settings
      solver_plugin: solver_plugins::CeresSolver

      # Map update rate
      map_update_interval: 5.0   # seconds between map saves

      # Resolution (meters per cell)
      resolution: 0.05           # 5cm cells (default)

      # Maximum range to use from LiDAR
      max_laser_range: 3.5       # match TurtleBot3 LiDAR range

      # Use simulation time
      use_sim_time: true
""")

print("\n✅ Exercise 2.1 Complete!")
print("=" * 80)



EXERCISE 2.1: Launching SLAM with TurtleBot3

📋 COMPLETE SLAM SETUP (4 terminals):
─────────────────────────────────────────────────────────────

  Terminal 1: Launch TurtleBot3 in world with obstacles
  ──────────────────────────────────────────────────────
  export TURTLEBOT3_MODEL=burger
  ros2 launch turtlebot3_gazebo turtlebot3_world.launch.py

  ⏱️  Wait until Gazebo fully loads (robot visible in world)


  Terminal 2: Launch slam_toolbox
  ──────────────────────────────────────────────────────
  ros2 launch slam_toolbox online_async_launch.py \
    slam_params_file:=/opt/ros/humble/share/turtlebot3_cartographer/config/turtlebot3_lds_2d.lua \
    use_sim_time:=true

  # Simpler version (uses default params):
  ros2 launch slam_toolbox online_async_launch.py \
    use_sim_time:=true

  # use_sim_time:=true = use Gazebo clock, not wall clock
  # CRITICAL: always set this for simulation!


  Terminal 3: Launch RViz2 with SLAM config
  ───────────────────────────────────────────────

In [7]:
# ==================================================
# EXERCISE 2.2: UNDERSTANDING THE MAP IN RVIZ
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.2: Understanding the Map in RViz2")
print("=" * 80)

print("""
📋 READING THE OCCUPANCY GRID MAP:
─────────────────────────────────────────────────────────────

  Color coding in RViz2:
  ┌─────────────────────────────────────────────────────┐
  │  ░░░░░░░░  Grey  = Unknown (not yet explored)       │
  │  ████████  Black = Occupied (wall or obstacle)      │
  │            White = Free space (robot can drive)     │
  └─────────────────────────────────────────────────────┘

  What good vs bad maps look like:

  GOOD MAP:                    BAD MAP:
  ─────────────                ──────────────────────
  ████████████                 ████  ████  (doubled walls)
  █          █                 ░░░░░░░░░░  (unexplored)
  █          █                 robot position uncertain
  █          █                 blurry obstacles
  ████████████


📋 LOOP CLOSURE EXPLAINED:
─────────────────────────────────────────────────────────────

  Without loop closure:
  Start ──→ drive around ──→ End (slightly off from Start)
  Map shows wall not quite closing → discontinuity

  With loop closure:
  Start ──→ drive around ──→ Return to Start
  SLAM recognizes "I've been here before!" →
  Corrects all accumulated odometry error →
  Map snaps together cleanly ✅

  You'll see this in RViz2 as a sudden map shift
  when the robot revisits a known area — that's loop closure!


📋 INSPECT THE MAP TOPIC:
─────────────────────────────────────────────────────────────
  ros2 topic echo /map --once

  Key fields:
  header:
    frame_id: "map"          ← coordinate frame
  info:
    resolution: 0.05         ← 5cm per cell
    width: 384               ← cells wide
    height: 384              ← cells tall
    origin:                  ← where (0,0) is in world coords
      position:
        x: -10.0
        y: -10.0
  data: [...]                ← flat array of cell values
                               -1=unknown, 0=free, 100=occupied
""")

print("\n✅ Exercise 2.2 Complete!")
print("=" * 80)


EXERCISE 2.2: Understanding the Map in RViz2

📋 READING THE OCCUPANCY GRID MAP:
─────────────────────────────────────────────────────────────

  Color coding in RViz2:
  ┌─────────────────────────────────────────────────────┐
  │  ░░░░░░░░  Grey  = Unknown (not yet explored)       │
  │  ████████  Black = Occupied (wall or obstacle)      │
  │            White = Free space (robot can drive)     │
  └─────────────────────────────────────────────────────┘

  What good vs bad maps look like:

  GOOD MAP:                    BAD MAP:
  ─────────────                ──────────────────────
  ████████████                 ████  ████  (doubled walls)
  █          █                 ░░░░░░░░░░  (unexplored)
  █          █                 robot position uncertain
  █          █                 blurry obstacles
  ████████████


📋 LOOP CLOSURE EXPLAINED:
─────────────────────────────────────────────────────────────

  Without loop closure:
  Start ──→ drive around ──→ End (slightly off from Start)
  

In [8]:
print("\n" + "=" * 80)
print("💾 PART 3: SAVING, LOADING & ANALYZING MAPS")
print("=" * 80)


💾 PART 3: SAVING, LOADING & ANALYZING MAPS


In [9]:
# ==================================================
# EXERCISE 3.1: SAVING AND LOADING MAPS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.1: Saving and Loading Maps")
print("=" * 80)

"""
📖 THEORY: Map Files

A saved map consists of two files:
- map.pgm  → Portable Gray Map image (the actual map pixels)
- map.yaml → Metadata (resolution, origin, thresholds)

These files are used by Nav2 for autonomous navigation (Day 109+).
"""

print("""
📋 SAVE THE MAP:
─────────────────────────────────────────────────────────────

  Method 1: nav2_map_server (standard)
  ──────────────────────────────────────
  # While SLAM is running, in a new terminal:
  ros2 run nav2_map_server map_saver_cli \\
    -f ~/ros2_ws/src/maps/my_map

  # This saves:
  # ~/ros2_ws/src/maps/my_map.pgm   ← map image
  # ~/ros2_ws/src/maps/my_map.yaml  ← metadata

  # Create maps directory first:
  mkdir -p ~/ros2_ws/src/maps


  Method 2: slam_toolbox service
  ──────────────────────────────────────
  ros2 service call /slam_toolbox/save_map \\
    slam_toolbox/srv/SaveMap \\
    "{name: {data: '/home/username/ros2_ws/src/maps/my_map'}}"


📋 MAP YAML FILE EXPLAINED:
─────────────────────────────────────────────────────────────
  # my_map.yaml — generated automatically by map_saver

  image: my_map.pgm           ← path to the .pgm file
  mode: trinary               ← how to interpret pixel values
  resolution: 0.05            ← meters per pixel (5cm)
  origin: [-10.0, -10.0, 0]  ← world coords of map bottom-left
  negate: 0                   ← 0=white is free, 1=black is free
  occupied_thresh: 0.65       ← probability threshold for occupied
  free_thresh: 0.25           ← probability threshold for free


📋 LOAD A SAVED MAP:
─────────────────────────────────────────────────────────────
  # Method 1: nav2_map_server (for navigation)
  ros2 run nav2_map_server map_server \\
    --ros-args -p yaml_filename:=~/ros2_ws/src/maps/my_map.yaml

  # Activate the map server:
  ros2 lifecycle set /map_server activate

  # The map is now published on /map topic
  # Open RViz2 to see it:
  rviz2
  # Add Map display → Topic: /map


  # Method 2: slam_toolbox localization mode
  ros2 launch slam_toolbox localization_launch.py \\
    map:=~/ros2_ws/src/maps/my_map.yaml \\
    use_sim_time:=true


📋 VIEW MAP AS IMAGE:
─────────────────────────────────────────────────────────────
  # Open the .pgm file directly:
  eog ~/ros2_ws/src/maps/my_map.pgm    # Eye of GNOME viewer
  # or
  display ~/ros2_ws/src/maps/my_map.pgm  # ImageMagick

  # Convert to PNG for sharing:
  convert ~/ros2_ws/src/maps/my_map.pgm my_map.png
""")

print("\n✅ Exercise 3.1 Complete!")
print("=" * 80)


EXERCISE 3.1: Saving and Loading Maps

📋 SAVE THE MAP:
─────────────────────────────────────────────────────────────

  Method 1: nav2_map_server (standard)
  ──────────────────────────────────────
  # While SLAM is running, in a new terminal:
  ros2 run nav2_map_server map_saver_cli \
    -f ~/ros2_ws/src/maps/my_map

  # This saves:
  # ~/ros2_ws/src/maps/my_map.pgm   ← map image
  # ~/ros2_ws/src/maps/my_map.yaml  ← metadata

  # Create maps directory first:
  mkdir -p ~/ros2_ws/src/maps


  Method 2: slam_toolbox service
  ──────────────────────────────────────
  ros2 service call /slam_toolbox/save_map \
    slam_toolbox/srv/SaveMap \
    "{name: {data: '/home/username/ros2_ws/src/maps/my_map'}}"


📋 MAP YAML FILE EXPLAINED:
─────────────────────────────────────────────────────────────
  # my_map.yaml — generated automatically by map_saver

  image: my_map.pgm           ← path to the .pgm file
  mode: trinary               ← how to interpret pixel values
  resolution: 0.05       

In [10]:
# ==================================================
# EXERCISE 3.2: DAY 108 SUMMARY
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.2: Day 108 Summary")
print("=" * 80)

print("""
📚 WHAT WE LEARNED TODAY:

✅ SLAM Fundamentals:
   • SLAM = Simultaneous Localization and Mapping
   • Solves the chicken-and-egg problem of robotics
   • Odometry alone drifts — SLAM corrects with LiDAR matching
   • Loop closure: recognizing visited places to correct errors
   • Output: occupancy grid map (free/occupied/unknown cells)

✅ slam_toolbox:
   • Recommended SLAM package for ROS2
   • online_async_launch.py for real-time mapping
   • use_sim_time:=true CRITICAL for Gazebo simulation
   • Subscribes to /scan and /tf
   • Publishes /map and map→odom transform

✅ Building a Map:
   • Drive slowly (0.1-0.15 m/s) for better quality
   • Cover all areas systematically
   • Return to start for loop closure
   • Watch map update live in RViz2
   • Grey=unknown, White=free, Black=occupied

✅ Saving and Loading Maps:
   • map_saver_cli saves .pgm + .yaml files
   • .yaml contains resolution, origin, thresholds
   • map_server loads and publishes saved maps
   • localization_launch.py uses existing map

💡 KEY INSIGHTS:
   1. SLAM is where your Week 14-15 skills converge
      LiDAR subscriber (Day 103) + ROS2 package (Day 102)
      + sensor processing pattern = SLAM input pipeline

   2. use_sim_time:=true is ALWAYS needed in simulation
      Without it, SLAM uses wall clock but Gazebo uses sim clock
      → timestamps don't match → SLAM fails silently

   3. The map is the robot's memory
      Once mapped, the robot can navigate autonomously.
      Nav2 (tomorrow) uses this map for path planning.

   4. Loop closure is magic
      Watching the map snap into alignment is one of the most
      satisfying moments in robotics — the robot "recognizes"
      where it is and corrects years of accumulated drift instantly.
""")

print("\n✅ Exercise 3.2 Complete!")
print("=" * 80)


EXERCISE 3.2: Day 108 Summary

📚 WHAT WE LEARNED TODAY:

✅ SLAM Fundamentals:
   • SLAM = Simultaneous Localization and Mapping
   • Solves the chicken-and-egg problem of robotics
   • Odometry alone drifts — SLAM corrects with LiDAR matching
   • Loop closure: recognizing visited places to correct errors
   • Output: occupancy grid map (free/occupied/unknown cells)

✅ slam_toolbox:
   • Recommended SLAM package for ROS2
   • online_async_launch.py for real-time mapping
   • use_sim_time:=true CRITICAL for Gazebo simulation
   • Subscribes to /scan and /tf
   • Publishes /map and map→odom transform

✅ Building a Map:
   • Drive slowly (0.1-0.15 m/s) for better quality
   • Cover all areas systematically
   • Return to start for loop closure
   • Watch map update live in RViz2
   • Grey=unknown, White=free, Black=occupied

✅ Saving and Loading Maps:
   • map_saver_cli saves .pgm + .yaml files
   • .yaml contains resolution, origin, thresholds
   • map_server loads and publishes saved m

In [11]:
print("\n" + "=" * 80)
print("🎯 DAY 108 COMPLETE! ✅")
print("=" * 80)

print("""
OBJECTIVES ACHIEVED:
   ✅ Understood SLAM concepts: localization + mapping simultaneously
   ✅ Launched slam_toolbox with TurtleBot3 in Gazebo
   ✅ Built a map by driving around the TurtleBot3 world
   ✅ Observed loop closure correcting odometry drift
   ✅ Saved map as .pgm + .yaml files
   ✅ Loaded saved map with map_server

💡 KEY FILES CREATED TODAY:
   ~/ros2_ws/src/maps/
   ├── my_map.pgm    ← occupancy grid map image
   └── my_map.yaml   ← map metadata

🎯 TOMORROW (DAY 109):
   - Nav2 (Navigation 2) stack
   - Autonomous point-to-point navigation
   - Path planning with the saved map
   - Set navigation goals in RViz2
   - Obstacle avoidance during navigation

Week 16 Progress: 42.9% (3/7 days complete)
Overall Progress: Day 108/168 (64.3% complete)
""")

print("=" * 80)


🎯 DAY 108 COMPLETE! ✅

OBJECTIVES ACHIEVED:
   ✅ Understood SLAM concepts: localization + mapping simultaneously
   ✅ Launched slam_toolbox with TurtleBot3 in Gazebo
   ✅ Built a map by driving around the TurtleBot3 world
   ✅ Observed loop closure correcting odometry drift
   ✅ Saved map as .pgm + .yaml files
   ✅ Loaded saved map with map_server

💡 KEY FILES CREATED TODAY:
   ~/ros2_ws/src/maps/
   ├── my_map.pgm    ← occupancy grid map image
   └── my_map.yaml   ← map metadata

🎯 TOMORROW (DAY 109):
   - Nav2 (Navigation 2) stack
   - Autonomous point-to-point navigation
   - Path planning with the saved map
   - Set navigation goals in RViz2
   - Obstacle avoidance during navigation

Week 16 Progress: 42.9% (3/7 days complete)
Overall Progress: Day 108/168 (64.3% complete)

